In [ ]:
# Esto es para construir los modelos
from tensorflow.keras.layers import Dense, Dropout, Input, Flatten, Conv2D ,MaxPooling2D, GlobalAveragePooling2D
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import plot_model

from keras.optimizers import Adam

from keras.losses import SparseCategoricalCrossentropy

# Esto es para entrenamientos
from keras.callbacks import EarlyStopping
from livelossplot import PlotLossesKeras

# Esto es para evaluación
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
import pandas as pd
# Graficos mmatrices
import plotly.express as px

import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
import os
import random
import numpy as np
import tensorflow as tf

SEED = 161105

def set_seeds(seed_value=SEED):
    # 1. Set the PYTHONHASHSEED environment variable at a fixed value
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    # 2. Set the python built-in pseudo-random generator at a fixed value
    random.seed(seed_value)
    # 3. Set the numpy pseudo-random generator at a fixed value
    np.random.seed(seed_value)
    # 4. Set the tensorflow pseudo-random generator at a fixed value
    tf.random.set_seed(seed_value)

In [ ]:
from src import data_loader as dl
from src import EDA

# Carga de datos

In [ ]:
X, Y = dl.load_data(fetch = False)

In [ ]:
for tag in X.keys():
    print(X[tag].shape, Y[tag].shape)

# Analisis Exploratorio

In [ ]:
fig1 = EDA.pixels_report(X = X)
fig1.show()

In [ ]:
fig2 = EDA.labels_report(Y = Y)
fig2.show()

In [ ]:
# Dummy Scaler

for tag in X.keys():
    X[tag] = X[tag] / 255.

In [ ]:
fig1 = EDA.pixels_report(X = X)
fig1.show()

In [ ]:
import numpy as np
import pandas as pd

def imbalance_report(Y, key="train"):
    y = np.ravel(Y[key])

    # Conteo por clase
    counts = pd.Series(y).value_counts().sort_index()

    # Asegurar que existan todas las clases (0 a n-1)
    all_classes = sorted(np.unique(y))
    counts = counts.reindex(all_classes, fill_value=0)

    # Estadísticas básicas
    total = counts.sum()
    percentages = counts / total * 100

    max_count = counts.max()
    min_count = counts.min()

    imbalance_ratio = max_count / max(min_count, 1)  # evitar división por cero

    # Ratio relativo por clase (vs clase mayoritaria)
    relative_ratio = counts / max_count

    # Construir reporte
    df = pd.DataFrame({
        "count": counts,
        "percentage": percentages.round(2),
        "ratio_vs_max": relative_ratio.round(3)
    })

    # Identificar clases críticas
    df["is_minority"] = df["ratio_vs_max"] < 0.3
    df["is_rare"] = df["ratio_vs_max"] < 0.1

    summary = {
        "total_samples": int(total),
        "n_classes": len(counts),
        "max_count": int(max_count),
        "min_count": int(min_count),
        "imbalance_ratio_max_min": round(imbalance_ratio, 2)
    }

    return df, summary

In [ ]:
import numpy as np

def compute_class_weights(Y, key="train"):
    y = np.ravel(Y[key])
    
    classes, counts = np.unique(y, return_counts=True)
    
    total = len(y)
    n_classes = len(classes)
    
    class_weights = {
        int(cls): total / (n_classes * count)
        for cls, count in zip(classes, counts)
    }
    
    return class_weights

In [ ]:
df_imbalance, summary = imbalance_report(Y, key="train")

display(df_imbalance)
print(summary)

In [ ]:
class_weights = compute_class_weights(Y, key="train")
print(class_weights)

In [ ]:

class disseas_classifier:
    def __init__(
        self,
        X,
        Y,
        attributes=None
    ):
        """
        CNN classifier configurable mediante diccionario de atributos.

        Parámetros esperados en attributes:
        {
            "conv_layers": [
                {
                    "filters": 16,
                    "kernel_size": (3, 3),
                    "activation": "relu",
                    "padding": "same",
                    "strides": (1, 1),
                    "pool_size": (2, 2),
                    "dropout": 0.2
                }
            ],
            "dense_layers": [
                {
                    "units": 32,
                    "activation": "relu",
                    "dropout": 0.2
                }
            ]
        }
        """
        self.X = X
        self.Y = Y
        self.attributes = attributes
        self.history = None

        input_shape = X["train"].shape[1:]
        n_classes = len(np.unique(np.ravel(Y["train"])))

        layers = [Input(shape=input_shape, name="input")]

        # Bloques convolucionales
        for index, conv_cfg in enumerate(attributes.get("conv_layers", []), start=1):
            layers.append(
                Conv2D(
                    filters = conv_cfg["filters"]
                    ,kernel_size = conv_cfg.get("kernel_size", (3, 3))
                    ,activation = conv_cfg.get("activation", "relu")
                    ,padding = conv_cfg.get("padding", "same")
                    ,strides=conv_cfg.get("strides", (1, 1))
                    ,name=f"conv{index}"
                )
            )

            pool_size = conv_cfg.get("pool_size", None)
            if pool_size is not None:
                layers.append(
                    MaxPooling2D(
                        pool_size = pool_size
                        ,name = f"pool{index}"
                    )
                )

            dropout_rate = conv_cfg.get("dropout", 0.0)
            if dropout_rate > 0:
                layers.append(
                    Dropout(dropout_rate, name = f"conv_dropout{index}")
                )

        # Reducción espacial para clasificación
        layers.append(GlobalAveragePooling2D(name = "global_avg_pool"))

        # Capas densas
        for index, dense_cfg in enumerate(attributes.get("dense_layers", []), start = 1):
            layers.append(
                Dense(
                    units = dense_cfg["units"]
                    ,activation=dense_cfg.get("activation", "relu")
                    ,name = f"hidden{index}"
                )
            )

            dropout_rate = dense_cfg.get("dropout", 0.0)
            if dropout_rate > 0:
                layers.append(
                    Dropout(dropout_rate, name=f"dense_dropout{index}")
                )

        # Salida
        layers.append(
            Dense(
                units = n_classes,
                activation = "softmax",
                name = "output"
            )
        )

        self.model = Sequential(layers=layers)

        try:
            plot_model(self.model, show_shapes=True)
        except Exception as e:
            print(f"Could not render plot_model: {e}")

    def fit(
        self,
        X,
        Y,
        train_tag,
        validation_tag,
        epochs=30,
        batch_size=16,
        learning_rate=1e-3,
        use_live_loss=False
    ):
        self.model.compile(
            optimizer = Adam(learning_rate=learning_rate),
            loss = SparseCategoricalCrossentropy(),
            metrics = ["accuracy"]
        )

        callbacks = [
            EarlyStopping(
                monitor = "val_loss",
                patience = 10,
                restore_best_weights = True
            )
        ]

        if use_live_loss:
            try:
                from livelossplot import PlotLossesKeras
                callbacks.append(PlotLossesKeras())
            except Exception as e:
                print(f"PlotLossesKeras could not be loaded: {e}")

        y_train = np.ravel(Y[train_tag])
        y_val = np.ravel(Y[validation_tag])

        self.history = self.model.fit(
            X[train_tag]
            ,y_train
            ,epochs = epochs
            ,batch_size = batch_size
            ,validation_data = (X[validation_tag], y_val)
            ,class_weight = class_weights 
            ,callbacks = callbacks
        )

        return self.history

    def metrics_report(self, X, Y):
        matrixes = {}

        for label in X.keys():
            y_true = np.ravel(Y[label])
            y_pred = self.model.predict(X[label], verbose=0)
            y_pred = np.argmax(y_pred, axis=1)

            conf_matrix = confusion_matrix(y_true, y_pred)
            conf_matrix_df = pd.DataFrame(conf_matrix)
            matrixes[label] = conf_matrix_df

            fig, ax = plt.subplots(figsize=(10, 8))
            sns.heatmap(
                conf_matrix_df,
                annot = True,
                fmt = "d",
                cmap = "Blues",
                cbar = False,
                ax = ax
            )
            ax.set_title(f"Confusion Matrix - {label}")
            ax.set_xlabel("Predicted")
            ax.set_ylabel("True")
            plt.show()

            report = classification_report(
                y_true,
                y_pred,
                output_dict=True,
                zero_division=np.nan
            )
            report = pd.DataFrame(report).T.round(2)

            print(label)
            display(report)

        return matrixes

In [ ]:
attributes = {
    "conv_layers": [
        {
            "filters": 16,
            "kernel_size": (3, 3),
            "activation": "relu",
            "padding": "same",
            "strides": (1, 1),
            "pool_size": (2, 2),
            "dropout": 0.2
        },
        {
            "filters": 32,
            "kernel_size": (3, 3),
            "activation": "relu",
            "padding": "same",
            "strides": (1, 1),
            "pool_size": (2, 2),
            "dropout": 0.1
        }
    ],
    "dense_layers": [
        {
            "units": 16,
            "activation": "relu",
            "dropout": 0.2
        },
        {
            "units": 8,
            "activation": "sigmoid",
            "dropout": 0.1
        }
    ]
}
model = disseas_classifier(X,Y,attributes = attributes, )


In [ ]:
model.fit(X,Y,'train','test', epochs = 100, use_live_loss = True)

In [ ]:
model.metrics_report(X,Y)